# People Identification in the Wild — **Voting (max count after floor)**

This notebook implements the **evidence / voting** idea from `FUTURE_IDEAS.md` (*Per-identity aggregation*, § “Overall support below threshold”) in a specific form:

1. **`MATCH_THRESHOLD`** (cosine distance): a gallery image counts as a **vote** for its identity if the distance to the probe crop is **≤** this value.
2. **`MIN_VOTES`**: an identity is **in the running** only if it has **at least** this many votes (e.g. **5** gallery images under threshold). Identities below the floor are ignored.
3. Among remaining candidates, pick the identity with the **largest** vote count (e.g. A:5, B:7 → **B**). **Tie-break:** same max count → pick the identity with the **smaller minimum** distance to the probe.

Sections **6.5** and **8** share one matcher and the same constants (single place to tune). Detection is **MTCNN** with raw boxes (no extra NMS), like the other variant notebooks.

Earlier sections still build `query_embedding` for demos; **ID assignment in 6.5 / 8 is gallery-only** using the rule above.


## 0) Install and import dependencies

We use:
- **DeepFace** for embeddings (face signatures)
- **RetinaFace / OpenCV backends via DeepFace** for face detection
- **OpenCV + Matplotlib** for visualization
- **NumPy/Pandas** for data handling

If you run this in Colab or a fresh environment, uncomment the install lines in the next cell.

In [ ]:
# -----------------------------
# INSTALL (UNCOMMENT IF NEEDED)
# -----------------------------
# !pip install deepface
# !pip install opencv-python matplotlib pandas tqdm

# -----------------------------
# IMPORTS
# -----------------------------
import os
import glob
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from deepface import DeepFace

# Make plots look cleaner
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

print("Imports loaded successfully.")

## 1) Locate and prepare dataset (including zip exploration)

This cell does three important things:

1. Finds any `.zip` files in the project folder (in case your dataset was added as a zip)
2. Extracts the zip into `open_data_set/` if needed
3. Confirms expected subfolders and prints image counts

This makes the notebook robust even if you share it with someone else who only has the zip file.

In [ ]:
# -----------------------------
# DATASET PATH DISCOVERY + ZIP EXTRACTION
# -----------------------------
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "open_data_set"

# If user dropped zip files in the project root, detect them automatically.
zip_files = sorted(PROJECT_ROOT.glob("*.zip"))

if zip_files:
    print("Found zip file(s):")
    for z in zip_files:
        print(" -", z.name)

    # Extract each zip into DATASET_ROOT. We keep this idempotent: extraction is safe to run again.
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    for z in zip_files:
        print(f"Extracting {z.name} -> {DATASET_ROOT}")
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(DATASET_ROOT)
else:
    print("No zip file found in project root. Assuming dataset is already extracted.")

# Expected folders from your dataset package.
expected_folders = [
    "photos_all",        # Crowd/group images
    "photos_all_faces",  # Pre-cropped faces for identities
    "portraits",         # Query-like single-person images
    "trio_cam",
    "trio_gp",
]

print("\nDataset root:", DATASET_ROOT)
for folder in expected_folders:
    folder_path = DATASET_ROOT / folder
    image_files = []
    if folder_path.exists():
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
            image_files.extend(folder_path.glob(ext))
    print(f"{folder:16s} -> exists={folder_path.exists()} | images={len(image_files)}")

## 2) Build metadata table from dataset files

We create a table of image paths and inferred identity labels.

- For this dataset, image names are like `a_gp_...jpg`, `k_gp_...jpg`.
- We infer identity from the first token (`a`, `b`, ..., `k`).
- This allows quick filtering for gallery faces and query faces.

In [ ]:
# -----------------------------
# BUILD IMAGE METADATA DATAFRAME
# -----------------------------
def list_images(folder: Path):
    """Return all image files from a folder with common extensions."""
    images = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        images.extend(folder.glob(ext))
    return sorted(images)


def infer_identity_from_filename(path: Path) -> str:
    """
    Infer person identity from filename prefix.
    Example: a_gp_4_ef_06.jpg -> 'a'

    We return uppercase ID labels (A, B, ..., K) to keep display clean.
    """
    token = path.stem.split("_")[0]
    return token.upper()

rows = []

# Gallery candidates: usually already-cropped face images.
gallery_dir = DATASET_ROOT / "photos_all_faces"
for p in list_images(gallery_dir):
    rows.append({
        "path": str(p),
        "split": "gallery",
        "identity": infer_identity_from_filename(p)
    })

# Query candidates: portrait images (single person).
portrait_dir = DATASET_ROOT / "portraits"
for p in list_images(portrait_dir):
    rows.append({
        "path": str(p),
        "split": "query_pool",
        "identity": infer_identity_from_filename(p)
    })

# Scene candidates: crowd/group images where we localize target.
scene_dir = DATASET_ROOT / "photos_all"
for p in list_images(scene_dir):
    rows.append({
        "path": str(p),
        "split": "scene_pool",
        "identity": p.stem.split("_")[0].upper()  # group token, not single identity
    })

meta_df = pd.DataFrame(rows)
print("Total records:", len(meta_df))
print(meta_df["split"].value_counts())

# Show available individual IDs from gallery/query pools.
id_df = meta_df[meta_df["split"].isin(["gallery", "query_pool"])]
print("\nUnique individual IDs:", sorted(id_df["identity"].unique()))
meta_df.head()

## 2.1) Explore dataset composition (important for objective quality)

Before training/matching, always inspect data balance.

This cell shows:
- how many gallery/query images each identity has
- a few random gallery samples

This is useful evidence for your report's data collection section.

In [ ]:
# -----------------------------
# DATASET EXPLORATION (COUNTS + SAMPLE IMAGES)
# -----------------------------
# Count identities in gallery and query pools.
count_table = (
    meta_df[meta_df["split"].isin(["gallery", "query_pool"])]
    .groupby(["split", "identity"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "identity"])
)

display(count_table)

# Plot identity distribution for gallery set.
gallery_counts = count_table[count_table["split"] == "gallery"]
plt.figure(figsize=(8, 4))
plt.bar(gallery_counts["identity"], gallery_counts["count"])
plt.title("Gallery image count per identity")
plt.xlabel("Identity")
plt.ylabel("Number of images")
plt.show()

# Show random gallery examples for quick visual inspection.
sample_gallery = meta_df[meta_df["split"] == "gallery"].sample(8, random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, (_, row) in zip(axes.flatten(), sample_gallery.iterrows()):
    img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"ID={row['identity']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3) Choose query image and scene image

- **Query image:** the person we want to find
- **Scene image:** crowd/group image where that person might appear

Tip: choose a query ID that is likely present in the scene token (for example, scene token `AB` likely contains A and B).

In [ ]:
# -----------------------------
# SELECT QUERY + SCENE
# -----------------------------
TARGET_ID = "A"  # Change this to any available ID: A, B, C, ...

# Pick first portrait for TARGET_ID if available, otherwise fallback to gallery image.
query_candidates = meta_df[(meta_df["identity"] == TARGET_ID) & (meta_df["split"] == "query_pool")]
if len(query_candidates) == 0:
    query_candidates = meta_df[(meta_df["identity"] == TARGET_ID) & (meta_df["split"] == "gallery")]

if len(query_candidates) == 0:
    raise ValueError(f"No query image found for TARGET_ID={TARGET_ID}")

query_path = query_candidates.iloc[0]["path"]

# Choose a scene image that likely contains target ID (based on group token in filename prefix).
scene_candidates = meta_df[(meta_df["split"] == "scene_pool") & (meta_df["identity"].str.contains(TARGET_ID, na=False))]
if len(scene_candidates) == 0:
    scene_candidates = meta_df[meta_df["split"] == "scene_pool"]

scene_path = scene_candidates.iloc[0]["path"]

print("Selected TARGET_ID:", TARGET_ID)
print("Query image:", query_path)
print("Scene image:", scene_path)

# Visual sanity check.
query_img = cv2.cvtColor(cv2.imread(query_path), cv2.COLOR_BGR2RGB)
scene_img = cv2.cvtColor(cv2.imread(scene_path), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(query_img)
axes[0].set_title(f"Query (ID={TARGET_ID})")
axes[0].axis("off")

axes[1].imshow(scene_img)
axes[1].set_title("Scene / Crowd image")
axes[1].axis("off")
plt.show()

## 4) Build gallery signatures (embeddings)

This corresponds to objective steps:
- Crop/use face images
- Extract signatures `s1, s2, ..., sn`

Here we use `photos_all_faces` as face crops, so we can run embedding extraction with `detector_backend='skip'` (faster and cleaner).

In [ ]:
# -----------------------------
# SIGNATURE EXTRACTION HELPERS
# -----------------------------
# ArcFace: best accuracy on standard/small datasets, ~0.05–0.15 cosine for same person.
# GhostFaceNet: faster/lighter but bundled DeepFace weights often give distances ~0.5
#               (random) on non-standard faces — verify self-test distances before using.
MODEL_NAME = "ArcFace"          # Best natively supported DeepFace model.
                                # Buffalo_L needs: pip install insightface onnxruntime
DISTANCE_METRIC = "cosine"

# Portrait images may show more than just a tight face crop.
# Use a detector for query images so the embedding covers only the face region.
QUERY_DETECTOR = "mtcnn"        # "skip" if portraits are already tight face crops


def get_embedding(image_path: str, model_name: str = MODEL_NAME, detector_backend: str = "skip"):
    """
    Return one embedding vector for a single image.

    Why detector_backend='skip'?
    - For pre-cropped face images (gallery/query), detection is unnecessary and can fail less often.
    """
    try:
        reps = DeepFace.represent(
            img_path=image_path,
            model_name=model_name,
            detector_backend=detector_backend,
            enforce_detection=False
        )
        if len(reps) == 0:
            return None
        # DeepFace returns a list of dicts; each dict has an 'embedding' vector.
        emb = np.array(reps[0]["embedding"], dtype=np.float32)
        return emb
    except Exception as e:
        print(f"Embedding failed for {image_path}: {e}")
        return None


def cosine_distance(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    """Compute cosine distance = 1 - cosine_similarity."""
    a_norm = np.linalg.norm(a) + eps
    b_norm = np.linalg.norm(b) + eps
    return float(1.0 - np.dot(a, b) / (a_norm * b_norm))


# Build gallery signatures (we use a subset per identity for speed in classroom runs).
gallery_df = meta_df[meta_df["split"] == "gallery"].copy()

# Optional downsample: keep up to N images per identity for faster runs.
MAX_PER_ID = 30
gallery_df = gallery_df.groupby("identity", group_keys=False).head(MAX_PER_ID).reset_index(drop=True)

embeddings = []
for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc="Building gallery embeddings"):
    emb = get_embedding(row["path"], detector_backend="skip")
    embeddings.append(emb)

gallery_df["embedding"] = embeddings
gallery_df = gallery_df[gallery_df["embedding"].notnull()].reset_index(drop=True)

print("Gallery size after embedding extraction:", len(gallery_df))
gallery_df.head()

## 5) Extract query signature `qs`

This is objective step 4: create the query embedding (`qs`) from the query face image.

In [ ]:
# -----------------------------
# QUERY SIGNATURE
# -----------------------------
# Portrait images may show shoulders/background around the face.
# Run QUERY_DETECTOR to crop the face region before embedding.
query_embedding = get_embedding(query_path, detector_backend=QUERY_DETECTOR)

if query_embedding is None:
    print(f"Warning: {QUERY_DETECTOR} found no face in query image — falling back to 'skip'.")
    query_embedding = get_embedding(query_path, detector_backend="skip")

if query_embedding is None:
    raise RuntimeError("Could not extract query embedding. Try another query image.")

print("Query embedding length:", len(query_embedding))

# Sanity check: nearest gallery match should be TARGET_ID with distance < ~0.25.
gallery_dists = gallery_df["embedding"].apply(lambda e: cosine_distance(query_embedding, e))
best_idx = int(gallery_dists.idxmin())
print(f"Nearest gallery match: identity={gallery_df.loc[best_idx, 'identity']}  "
      f"cosine_distance={gallery_dists[best_idx]:.4f}")
if gallery_dists[best_idx] > 0.35:
    print("⚠  High distance — consider switching MODEL_NAME to 'Buffalo_L'/'ArcFace' or checking query image quality.")

## 5.1) Visual test case — `cd_gp_0_eo_12.JPG`

This scene contains **4 visible people**, but the filename token is `CD` — meaning only **C and D** are expected subjects.
The other 2 people are **bystanders** not in the gallery, or different identities. This tests whether the pipeline:
1. Detects all 4 faces
2. Correctly identifies C and D
3. Rejects the other 2 as non-matches (gallery distance above threshold)

In [ ]:
# ---------------------------------------------------------
# VISUAL TEST CASE: cd_gp_0_eo_12.JPG
# Detect → crop → embed → find nearest gallery match
# Show each scene crop next to its best gallery match image
# ---------------------------------------------------------
# MTCNN builds its own image pyramid internally, so no manual
# upscaling is needed — we pass the original image directly.

TEST_SCENE = str(DATASET_ROOT / "photos_all" / "cd_gp_0_eo_12.JPG")
TEST_MATCH_THRESHOLD = 0.15

test_img = cv2.imread(TEST_SCENE)
if test_img is None:
    raise FileNotFoundError(f"Cannot read test image: {TEST_SCENE}")

_h0, _w0 = test_img.shape[:2]
print(f"Original: {_w0}x{_h0}")

try:
    _raw_faces = DeepFace.extract_faces(
        img_path=test_img, detector_backend="mtcnn",
        enforce_detection=True, align=True,
    )
except Exception:
    _raw_faces = []

_test_boxes = []
for _item in _raw_faces:
    _a = _item["facial_area"]
    _x = max(0, min(int(_a["x"]), _w0 - 1))
    _y = max(0, min(int(_a["y"]), _h0 - 1))
    _w = max(1, min(int(_a["w"]), _w0 - _x))
    _h = max(1, min(int(_a["h"]), _h0 - _y))
    if _w >= 10 and _h >= 10:
        _test_boxes.append({"x": _x, "y": _y, "w": _w, "h": _h})

print(f"Scene: cd_gp_0_eo_12.JPG | Raw detections: {len(_test_boxes)}")

# Show the scene with detection boxes.
_vis = cv2.cvtColor(test_img.copy(), cv2.COLOR_BGR2RGB)
for _i, _b in enumerate(_test_boxes):
    cv2.rectangle(_vis, (_b["x"], _b["y"]),
                  (_b["x"]+_b["w"], _b["y"]+_b["h"]), (0, 255, 0), 2)
    cv2.putText(_vis, str(_i), (_b["x"]+2, _b["y"]+14),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
plt.figure(figsize=(12, 7))
plt.imshow(_vis)
plt.title(f"MTCNN detections ({len(_test_boxes)}) — cd_gp_0_eo_12.JPG (expected: C, D)")
plt.axis("off")
plt.show()

# --- For each detection: embed, find nearest gallery image, display side-by-side ---
_n = len(_test_boxes)
if _n == 0:
    print("No faces detected.")
else:
    fig, axes = plt.subplots(_n, 2, figsize=(6, 3 * _n))
    if _n == 1:
        axes = [axes]

    for _i, _b in enumerate(_test_boxes):
        crop_bgr = test_img[_b["y"]:_b["y"]+_b["h"], _b["x"]:_b["x"]+_b["w"]]
        if crop_bgr.size == 0:
            continue
        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

        # Embed the scene crop.
        try:
            _rep = DeepFace.represent(
                img_path=crop_rgb, model_name=MODEL_NAME,
                detector_backend="skip", enforce_detection=False,
            )
            if not _rep:
                raise ValueError("empty")
            _emb = np.array(_rep[0]["embedding"], dtype=np.float32)
        except Exception as _e:
            axes[_i][0].imshow(crop_rgb)
            axes[_i][0].set_title(f"Face #{_i} — embed failed")
            axes[_i][0].axis("off")
            axes[_i][1].axis("off")
            continue

        # Find nearest gallery image.
        _dists = gallery_df["embedding"].apply(lambda e: cosine_distance(_emb, e))
        _best_idx = int(_dists.idxmin())
        _best_dist = float(_dists.loc[_best_idx])
        _pred_id = gallery_df.loc[_best_idx, "identity"]
        _gallery_path = gallery_df.loc[_best_idx, "path"]

        _is_match = _best_dist <= TEST_MATCH_THRESHOLD
        _match_label = "MATCH" if _is_match else "NO MATCH"
        _color = "green" if _is_match else "red"

        # Load the gallery image that matched.
        _gallery_img = cv2.cvtColor(cv2.imread(_gallery_path), cv2.COLOR_BGR2RGB)

        # Left: scene crop.
        axes[_i][0].imshow(crop_rgb)
        axes[_i][0].set_title(f"Scene face #{_i}", fontsize=10)
        axes[_i][0].axis("off")

        # Right: nearest gallery match.
        axes[_i][1].imshow(_gallery_img)
        axes[_i][1].set_title(
            f"→ {_pred_id} (dist={_best_dist:.4f}) [{_match_label}]",
            fontsize=10, color=_color,
        )
        axes[_i][1].axis("off")

        print(f"Face #{_i}: predicted={_pred_id}  gallery_dist={_best_dist:.4f}  {_match_label}")

    fig.suptitle("MTCNN + ArcFace — cd_gp_0_eo_12.JPG\n"
                 f"Expected identities: C, D  |  threshold={TEST_MATCH_THRESHOLD}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


Tests **every combination** of:
- **Detector backends**: opencv, mtcnn, retinaface, ssd, mediapipe
- **Upscale factors**: 1.0, 1.5, 2.0, 3.0
- **Recognition models**: ArcFace, Facenet512, SFace, GhostFaceNet, Dlib

For each combo, shows the scene crop alongside the gallery image the model matched it to. This reveals which detector+recognizer pairing works best for this scene.

In [ ]:
# (Section removed)


This diagnostic section shows exactly what each detector backend returns at each scale:

- Annotated scene image (bounding boxes)
- Cropped face thumbnails

Use this to compare detector behavior and verify whether a backend/scale is genuinely finding faces or not.

In [ ]:
# (Section removed — using MTCNN as fixed detector)

## 6.4) Fixed test setup: MTCNN detector

This section benchmarks recognition models using MTCNN as the fixed detector.
MTCNN builds its own image pyramid internally, so no manual upscaling is needed.

**Detection output:** we keep **all** MTCNN boxes that pass a minimum size check. Extra post-processing NMS was removed as redundant for this project.

Use this when you want a fair same-detector comparison across recognition models.


In [ ]:
# -----------------------------
# FIXED DETECTOR (MTCNN) + RECOGNITION MODEL TEST
# -----------------------------
# MTCNN has a built-in image pyramid so no manual upscaling is needed.
import time

scene_bgr = cv2.imread(scene_path)
if scene_bgr is None:
    raise FileNotFoundError(f"Could not read scene image: {scene_path}")
scene_h, scene_w = scene_bgr.shape[:2]

FIXED_DETECTOR_BACKEND = "mtcnn"
print(f"Scene: {scene_w}x{scene_h} | Detector: {FIXED_DETECTOR_BACKEND}")

MODEL_CANDIDATES = [
    "ArcFace",
    "Buffalo_L",
    "GhostFaceNet",
    "SFace",
    "Facenet512",
    "Dlib",
]


def _safe_query_embedding(model_name: str):
    try:
        t0 = time.time()
        rep = DeepFace.represent(
            img_path=query_path,
            model_name=model_name,
            detector_backend="skip",
            enforce_detection=False,
        )
        if len(rep) == 0:
            return None, None, "empty_embedding"
        emb = np.array(rep[0]["embedding"], dtype=np.float32)
        return emb, time.time() - t0, None
    except Exception as e:
        return None, None, str(e)


h0, w0 = scene_bgr.shape[:2]
try:
    raw_faces = DeepFace.extract_faces(
        img_path=scene_bgr,
        detector_backend=FIXED_DETECTOR_BACKEND,
        enforce_detection=True,
        align=True,
    )
except Exception:
    raw_faces = []

raw_boxes = []
for item in raw_faces:
    area = item["facial_area"]
    x = max(0, min(int(area["x"]), w0 - 1))
    y = max(0, min(int(area["y"]), h0 - 1))
    w = max(1, min(int(area["w"]), w0 - x))
    h = max(1, min(int(area["h"]), h0 - y))
    if w >= 10 and h >= 10:
        raw_boxes.append({"x": x, "y": y, "w": w, "h": h})

# Use MTCNN boxes as-is (no extra NMS). Sanity-check geometry.
fixed_boxes = list(raw_boxes)
for b in fixed_boxes:
    assert b["w"] >= 10 and b["h"] >= 10
    assert 0 <= b["x"] < w0 and 0 <= b["y"] < h0
    assert b["x"] + b["w"] <= w0 and b["y"] + b["h"] <= h0

print(f"Fixed detector setup -> backend={FIXED_DETECTOR_BACKEND}")
print(f"MTCNN detections kept (no post-NMS): {len(fixed_boxes)}")

if len(fixed_boxes) == 0:
    raise RuntimeError("MTCNN found 0 faces on this scene. Try another scene image.")

# Show fixed detector boxes.
vis = cv2.cvtColor(scene_bgr.copy(), cv2.COLOR_BGR2RGB)
for b in fixed_boxes:
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

plt.figure(figsize=(10, 5))
plt.imshow(vis)
plt.title(f"Fixed detector boxes | {FIXED_DETECTOR_BACKEND}")
plt.axis("off")
plt.show()

# Crop scene faces once from fixed detections.
scene_crops_rgb = []
for b in fixed_boxes:
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    crop_bgr = scene_bgr[y:y+h, x:x+w]
    if crop_bgr.size == 0:
        continue
    scene_crops_rgb.append(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

# Benchmark recognition models under fixed detector setup.
rows = []
for model_name in MODEL_CANDIDATES:
    q_emb, q_time, q_err = _safe_query_embedding(model_name)
    if q_emb is None:
        rows.append({
            "model": model_name,
            "best_query_distance": np.nan,
            "num_scene_faces_used": 0,
            "query_embed_sec": None,
            "scene_embed_total_sec": None,
            "detector_backend": FIXED_DETECTOR_BACKEND,
            "error": q_err,
        })
        continue

    best_dist = 999.0
    used = 0
    t0 = time.time()

    for crop_rgb in scene_crops_rgb:
        try:
            rep = DeepFace.represent(
                img_path=crop_rgb,
                model_name=model_name,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            dist = cosine_distance(q_emb, emb)
            best_dist = min(best_dist, dist)
            used += 1
        except Exception:
            continue

    rows.append({
        "model": model_name,
        "best_query_distance": best_dist if used > 0 else np.nan,
        "num_scene_faces_used": used,
        "query_embed_sec": round(q_time, 3) if q_time is not None else None,
        "scene_embed_total_sec": round(time.time() - t0, 3),
        "detector_backend": FIXED_DETECTOR_BACKEND,
        "error": None if used > 0 else "no_valid_scene_embeddings",
    })

fixed_model_bench_df = pd.DataFrame(rows).sort_values("best_query_distance", na_position="last")
print("Recognition model comparison with fixed detector setup:")
display(fixed_model_bench_df)

valid = fixed_model_bench_df[fixed_model_bench_df["best_query_distance"].notnull()]
if len(valid) > 0:
    print(f"Best model in this fixed test: {valid.iloc[0]['model']}")


## 6.5.0) Matching rules (`FUTURE_IDEAS.md`)

Run the **next code cell** once. It defines **`MATCH_THRESHOLD`**, **`MIN_VOTES`**, and **`match_gallery_vote_max`** for sections **6.5** and **8**.


In [ ]:
# -----------------------------
# 6.5.0) Matching rules — vote COUNT with minimum floor (FUTURE_IDEAS.md)
# -----------------------------
# Tune only here for sections 6.5 and 8.

MATCH_THRESHOLD = 0.15   # Cosine distance: gallery row counts as a "vote" if distance <= this.
MIN_VOTES = 5            # Identity must have at least this many votes to be a candidate; then pick max count.


def match_gallery_vote_max(gallery_df: pd.DataFrame, probe_emb: np.ndarray) -> dict:
    """
    Voting with a participation floor (FUTURE_IDEAS.md — evidence across gallery images).

    - Count, per identity, how many gallery images have distance <= MATCH_THRESHOLD.
    - Keep only identities with count >= MIN_VOTES.
    - If none qualify, no match.
    - Else choose the identity with the highest count (e.g. A:5, B:7 -> B).
    - Tie on count: pick the identity whose *minimum* distance to the probe is smaller.
    """
    dists = gallery_df["embedding"].apply(lambda e: cosine_distance(probe_emb, e))
    g = gallery_df.assign(_d=dists)
    under = g["_d"] <= MATCH_THRESHOLD

    def _diag_row():
        j = int(dists.idxmin())
        return {
            "predicted_id": None,
            "accepted": False,
            "vote_count": 0,
            "runner_up_id": None,
            "second_best_count": 0,
            "nearest_distance": float(dists.loc[j]),
            "min_dist_global": float(dists.min()),
            "gallery_path": g.loc[j, "path"],
        }

    if under.sum() == 0:
        return _diag_row()

    vc = g.loc[under].groupby("identity").size()
    candidates = vc[vc >= MIN_VOTES]
    if len(candidates) == 0:
        out = _diag_row()
        out["vote_count"] = int(vc.max()) if len(vc) else 0
        return out

    top = int(candidates.max())
    finalists = candidates[candidates == top].index.tolist()

    def _min_d(ident):
        return float(g.loc[g["identity"] == ident, "_d"].min())

    pred_id = min(finalists, key=_min_d)
    sub = g[g["identity"] == pred_id]
    j_best = int(sub["_d"].idxmin())
    nearest = float(sub["_d"].min())

    rest = candidates.drop(pred_id, errors="ignore").sort_values(ascending=False)
    runner_up_id = None
    second_best_count = 0
    if len(rest) > 0:
        runner_up_id = rest.index[0]
        second_best_count = int(rest.iloc[0])

    return {
        "predicted_id": pred_id,
        "accepted": True,
        "vote_count": top,
        "runner_up_id": runner_up_id,
        "second_best_count": second_best_count,
        "nearest_distance": nearest,
        "min_dist_global": float(dists.min()),
        "gallery_path": g.loc[j_best, "path"],
    }


# --- Toy sanity: floor filters low-count IDs; winner is max among qualifiers ---
_toy = pd.DataFrame(
    {
        "identity": ["A"] * 5 + ["B"] * 7,
        "path": [f"a{i}" for i in range(5)] + [f"b{i}" for i in range(7)],
        "embedding": [np.array([1.0, 0.0], np.float32)] * 5 + [np.array([0.92, 0.392], np.float32)] * 7,
    }
)
_probe = np.array([1.0, 0.0], np.float32)
_r = match_gallery_vote_max(_toy, _probe)
assert _r["predicted_id"] == "B" and _r["vote_count"] == 7, "B has more votes when both meet MIN_VOTES"

_toy2 = pd.DataFrame(
    {
        "identity": ["A"] * 4 + ["B"] * 7,
        "path": [f"a{i}" for i in range(4)] + [f"b{i}" for i in range(7)],
        "embedding": [np.array([1.0, 0.0], np.float32)] * 4 + [np.array([0.92, 0.392], np.float32)] * 7,
    }
)
_r2 = match_gallery_vote_max(_toy2, _probe)
assert _r2["predicted_id"] == "B" and _r2["accepted"] is True, "A below floor; only B qualifies"

print("Sanity OK: vote-max with MIN_VOTES floor matches FUTURE_IDEAS voting scenario.")
del _toy, _toy2, _probe, _r, _r2


## 6.5) Show detected face images and recognized IDs (vote max)

For each face crop we count votes per identity (distance **≤ `MATCH_THRESHOLD`**), keep only identities with **≥ `MIN_VOTES`** votes, then pick the **largest** count.

**Prerequisite:** section **6.5.0** above.


In [ ]:
# -----------------------------
# VISUALIZE DETECTED FACES + PREDICTED IDs
# -----------------------------
if "match_gallery_vote_max" not in globals():
    raise RuntimeError("Run section 6.5.0 first (defines MATCH_THRESHOLD, MIN_VOTES, match_gallery_vote_max).")

# Pick recognition model for this visualization.
if "fixed_model_bench_df" in globals() and len(fixed_model_bench_df) > 0:
    valid_rows = fixed_model_bench_df[fixed_model_bench_df["best_query_distance"].notnull()]
    RECOG_MODEL_FOR_VIS = valid_rows.iloc[0]["model"] if len(valid_rows) > 0 else MODEL_NAME
else:
    RECOG_MODEL_FOR_VIS = MODEL_NAME

print(f"Using recognition model for ID visualization: {RECOG_MODEL_FOR_VIS}")

# Choose boxes source:
if "fixed_boxes" in globals() and len(fixed_boxes) > 0:
    vis_boxes = [{"x": b["x"], "y": b["y"], "w": b["w"], "h": b["h"]} for b in fixed_boxes]
elif "scene_faces" in globals() and len(scene_faces) > 0:
    vis_boxes = [
        {
            "x": int(f["facial_area"]["x"]),
            "y": int(f["facial_area"]["y"]),
            "w": int(f["facial_area"]["w"]),
            "h": int(f["facial_area"]["h"]),
        }
        for f in scene_faces
    ]
else:
    raise RuntimeError("No detections available. Run section 6.4 first (MTCNN → fixed_boxes).")

# Build gallery embeddings for the SAME recognition model.
if "gallery_embeddings_by_model" not in globals():
    gallery_embeddings_by_model = {}

if (
    RECOG_MODEL_FOR_VIS not in gallery_embeddings_by_model
    or "path" not in gallery_embeddings_by_model[RECOG_MODEL_FOR_VIS].columns
):
    print("Building gallery embeddings for model:", RECOG_MODEL_FOR_VIS)
    g_rows = []
    for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc="Gallery embeddings (ID vis)"):
        try:
            rep = DeepFace.represent(
                img_path=row["path"],
                model_name=RECOG_MODEL_FOR_VIS,
                detector_backend="skip",
                enforce_detection=False,
            )
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            g_rows.append({"identity": row["identity"], "embedding": emb, "path": row["path"]})
        except Exception:
            continue

    if len(g_rows) == 0:
        raise RuntimeError("Failed to build gallery embeddings for visualization model.")

    gallery_embeddings_by_model[RECOG_MODEL_FOR_VIS] = pd.DataFrame(g_rows)

gallery_vis_df = gallery_embeddings_by_model[RECOG_MODEL_FOR_VIS]

# Gallery structure sanity (voting needs multiple images per ID to be meaningful).
_id_counts = gallery_vis_df["identity"].value_counts()
print("Gallery images per identity (vis model):")
display(_id_counts.to_frame("n_images"))
if _id_counts.min() < 2:
    print("WARNING: some identities have only one gallery image — voting cannot outvote outliers for those IDs.")

# Predict ID for each detected box (vote-max voting).
id_results = []
for i, b in enumerate(vis_boxes):
    x, y, w, h = b["x"], b["y"], b["w"], b["h"]
    crop_bgr = scene_bgr[y:y+h, x:x+w]
    if crop_bgr.size == 0:
        continue

    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)

    try:
        rep = DeepFace.represent(
            img_path=crop_rgb,
            model_name=RECOG_MODEL_FOR_VIS,
            detector_backend="skip",
            enforce_detection=False,
        )
        if len(rep) == 0:
            continue
        emb = np.array(rep[0]["embedding"], dtype=np.float32)
    except Exception:
        continue

    m = match_gallery_vote_max(gallery_vis_df, emb)
    pred_disp = m["predicted_id"] if m["accepted"] else "UNKNOWN"
    is_match = bool(m["accepted"])

    id_results.append(
        {
            "det_idx": i,
            "predicted_id": pred_disp,
            "raw_winner_id": m["predicted_id"],
            "vote_count": m["vote_count"],
            "runner_up_id": m["runner_up_id"],
            "second_best_count": m["second_best_count"],
            "nearest_distance": m["nearest_distance"],
            "min_dist_global": m["min_dist_global"],
            "is_match": is_match,
            "gallery_path": m["gallery_path"],
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "crop_rgb": crop_rgb,
        }
    )

id_results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ["crop_rgb", "gallery_path"]}
    for r in id_results
])
print("Detected faces with predicted IDs (vote-max):")
display(id_results_df.sort_values("nearest_distance"))

if len(id_results) == 0:
    raise RuntimeError("No valid ID predictions produced.")

assert MIN_VOTES >= 1  # e.g. 5 — tune in 6.5.0 only; batch eval reads the same globals

# 1) Annotated full scene with all predicted IDs.
scene_annot = scene_bgr.copy()
for r in id_results:
    x, y, w, h = r["x"], r["y"], r["w"], r["h"]
    label = f"ID={r['predicted_id']} votes={r['vote_count']} d={r['nearest_distance']:.3f}"
    cv2.rectangle(scene_annot, (x, y), (x + w, y + h), (0, 255, 0), 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 2)
    cv2.rectangle(scene_annot, (x, max(0, y - 22)), (x + tw + 6, y), (0, 255, 0), -1)
    cv2.putText(scene_annot, label, (x + 3, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(scene_annot, cv2.COLOR_BGR2RGB))
plt.title(f"All detected faces with predicted IDs — voting ({RECOG_MODEL_FOR_VIS})")
plt.axis("off")
plt.show()

# Save annotated all-ID image.
all_ids_output_path = str(PROJECT_ROOT / "output_all_detected_faces_with_ids.jpg")
cv2.imwrite(all_ids_output_path, scene_annot)
print("Saved annotated all-ID output:", all_ids_output_path)

# 2) Side-by-side: crop vs supporting gallery image
id_results_sorted = sorted(id_results, key=lambda r: r["nearest_distance"])
_n = len(id_results_sorted)
fig, axes = plt.subplots(_n, 2, figsize=(8, 3.2 * _n))
if _n == 1:
    axes = [axes]

for _i, r in enumerate(id_results_sorted):
    gp = r["gallery_path"]
    gallery_bgr = cv2.imread(gp) if gp and not str(gp).startswith("toy_") else None
    gallery_rgb = cv2.cvtColor(gallery_bgr, cv2.COLOR_BGR2RGB) if gallery_bgr is not None else r["crop_rgb"]

    match_label = "MATCH" if r["is_match"] else "NO MATCH"
    title_color = "green" if r["is_match"] else "red"

    axes[_i][0].imshow(r["crop_rgb"])
    axes[_i][0].set_title(f"Scene Det#{r['det_idx']}")
    axes[_i][0].axis("off")

    axes[_i][1].imshow(gallery_rgb)
    axes[_i][1].set_title(
        f"→ {r['predicted_id']} (d={r['nearest_distance']:.4f}, votes={r['vote_count']}) [{match_label}]",
        color=title_color,
    )
    axes[_i][1].axis("off")

    print(
        f"Det#{r['det_idx']}: predicted={r['predicted_id']} "
        f"d={r['nearest_distance']:.4f} votes={r['vote_count']} {match_label}"
    )

fig.suptitle(
    f"Scene crops vs gallery (vote-max) ({RECOG_MODEL_FOR_VIS}) | T={MATCH_THRESHOLD}, min_votes={MIN_VOTES}",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()


## 8) Full per-scene evaluation (gallery **vote max**), TP/FP, precision/recall, CSV

For each scene image:

- Detect with **MTCNN** (section 6.4; no extra NMS).
- Each crop: **`match_gallery_vote_max`** — same rule as section 6.5.
- Compare to expected IDs from the filename token.
- Save **`batch_evaluation_results.csv`**.

**Prerequisite:** section **6.5.0** (`match_gallery_vote_max`, thresholds).


In [ ]:
# -----------------------------
# FULL PER-SCENE EVALUATION WITH GROUND TRUTH + TP/FP + CSV
# -----------------------------
if "match_gallery_vote_max" not in globals():
    raise RuntimeError("Run section 6.5.0 first — batch eval must use match_gallery_vote_max like section 6.5.")

# Batch eval uses MODEL_NAME for a stable, user-controlled embedding model (independent of 6.5 vis model).
# Acceptance still uses MATCH_THRESHOLD / MIN_VOTES from 6.5.0 — keep them in sync.
assert "MATCH_THRESHOLD" in globals() and "MIN_VOTES" in globals()

scene_paths = meta_df[meta_df["split"] == "scene_pool"]["path"].tolist()


def token_to_expected_ids(token):
    """e.g. 'EF' -> {'E','F'}, 'IJK' -> {'I','J','K'}"""
    return set(list(token.upper()))


def detect_and_crop(scene_img):
    """
    MTCNN via DeepFace; return RGB crops + boxes. No post-NMS (same as section 6.4).
    """
    h0, w0 = scene_img.shape[:2]
    try:
        raw_faces = DeepFace.extract_faces(
            img_path=scene_img,
            detector_backend=FIXED_DETECTOR_BACKEND,
            enforce_detection=True,
            align=True,
        )
    except Exception:
        raw_faces = []

    raw_boxes = []
    for item in raw_faces:
        area = item["facial_area"]
        x = max(0, min(int(area["x"]), w0 - 1))
        y = max(0, min(int(area["y"]), h0 - 1))
        w = max(1, min(int(area["w"]), w0 - x))
        h = max(1, min(int(area["h"]), h0 - y))
        if w >= 10 and h >= 10:
            raw_boxes.append({"x": x, "y": y, "w": w, "h": h})

    boxes = list(raw_boxes)
    for b in boxes:
        assert b["w"] >= 10 and b["h"] >= 10
        assert 0 <= b["x"] < w0 and 0 <= b["y"] < h0
        assert b["x"] + b["w"] <= w0 and b["y"] + b["h"] <= h0

    crops = []
    for b in boxes:
        crop_bgr = scene_img[b["y"]:b["y"]+b["h"], b["x"]:b["x"]+b["w"]]
        if crop_bgr.size == 0:
            continue
        crops.append({
            "crop_rgb": cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB),
            "x": b["x"], "y": b["y"], "w": b["w"], "h": b["h"],
        })
    return crops


if "gallery_embeddings_by_model" not in globals():
    gallery_embeddings_by_model = {}

if MODEL_NAME not in gallery_embeddings_by_model:
    g_rows = []
    for _, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc="Gallery embeddings (batch eval)"):
        try:
            rep = DeepFace.represent(img_path=row["path"], model_name=MODEL_NAME,
                                     detector_backend="skip", enforce_detection=False)
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
            g_rows.append({"identity": row["identity"], "embedding": emb, "path": row["path"]})
        except Exception:
            continue
    gallery_embeddings_by_model[MODEL_NAME] = pd.DataFrame(g_rows)

gallery_eval_df = gallery_embeddings_by_model[MODEL_NAME]

_id_ce = gallery_eval_df["identity"].value_counts()
print("Gallery images per identity (batch model):")
display(_id_ce.to_frame("n_images"))
if _id_ce.min() < 2:
    print("WARNING: some identities have only one gallery image — voting is degenerate for those IDs.")

scene_rows = []
face_rows = []

for sp in tqdm(scene_paths, desc="Batch evaluation"):
    token = Path(sp).stem.split("_")[0].upper()
    expected_ids = token_to_expected_ids(token)

    scene_img = cv2.imread(sp)
    if scene_img is None:
        scene_rows.append({
            "scene_path": sp, "scene_token": token,
            "expected_ids": str(sorted(expected_ids)),
            "num_detected": 0, "all_predicted_ids": "",
            "true_positives": 0, "false_positives": 0,
            "precision": np.nan, "recall": np.nan, "error": "could_not_read",
        })
        continue

    crops = detect_and_crop(scene_img)

    predicted_ids_in_scene = []

    for crop_info in crops:
        crop_rgb = crop_info["crop_rgb"]
        try:
            rep = DeepFace.represent(img_path=crop_rgb, model_name=MODEL_NAME,
                                     detector_backend="skip", enforce_detection=False)
            if len(rep) == 0:
                continue
            emb = np.array(rep[0]["embedding"], dtype=np.float32)
        except Exception:
            continue

        m = match_gallery_vote_max(gallery_eval_df, emb)
        pred_id = m["predicted_id"] if m["accepted"] else None

        if m["accepted"] and pred_id is not None:
            predicted_ids_in_scene.append(pred_id)

        face_rows.append({
            "scene_path": sp,
            "scene_token": token,
            "expected_ids": str(sorted(expected_ids)),
            "predicted_id": pred_id,
            "vote_count": m["vote_count"],
            "runner_up_id": m["runner_up_id"],
            "second_best_count": m["second_best_count"],
            "nearest_gallery_distance": m["nearest_distance"],
            "accepted": m["accepted"],
            "is_true_positive": (pred_id in expected_ids) if pred_id is not None else False,
            "x": crop_info["x"], "y": crop_info["y"],
            "w": crop_info["w"], "h": crop_info["h"],
        })

    predicted_set = set(predicted_ids_in_scene)
    tp = len(predicted_set & expected_ids)
    fp = len(predicted_set - expected_ids)
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall    = tp / len(expected_ids) if len(expected_ids) > 0 else np.nan

    scene_rows.append({
        "scene_path": sp,
        "scene_token": token,
        "expected_ids": str(sorted(expected_ids)),
        "num_detected": len(crops),
        "all_predicted_ids": str(sorted(predicted_set)),
        "true_positives": tp,
        "false_positives": fp,
        "precision": round(precision, 3) if not np.isnan(precision) else np.nan,
        "recall":    round(recall,    3) if not np.isnan(recall)    else np.nan,
        "error": None,
    })

scene_eval_df = pd.DataFrame(scene_rows)
face_eval_df  = pd.DataFrame(face_rows)

csv_path = str(PROJECT_ROOT / "batch_evaluation_results.csv")
scene_eval_df.to_csv(csv_path, index=False)
print(f"Results saved to: {csv_path}")

print(f"\nTotal scenes evaluated: {len(scene_eval_df)}")
print(f"Scenes with ≥1 TP:     {(scene_eval_df['true_positives'] > 0).sum()}")
print(f"Mean precision:         {scene_eval_df['precision'].mean():.3f}")
print(f"Mean recall:            {scene_eval_df['recall'].mean():.3f}")

print("\nPer-scene results (sorted by recall desc):")
display(scene_eval_df.sort_values("recall", ascending=False).head(30))

print("\nPer group-token summary:")
group_summary = (
    scene_eval_df.groupby("scene_token")
    .agg(
        scenes=("scene_path", "count"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_tp=("true_positives", "mean"),
        mean_fp=("false_positives", "mean"),
    )
    .round(3)
    .reset_index()
)
display(group_summary)


## 10) Objective checklist (mapping)

- **Task 1 (Data collection):** Dataset discovered/extracted and explored (`open_data_set`)
- **Task 2.1:** Detect all faces in input image (`DeepFace.extract_faces`, MTCNN in 6.4; no extra NMS)
- **Task 2.2:** Crop detected faces (per bounding box)
- **Task 2.3:** Extract signatures for all cropped faces (`DeepFace.represent`)
- **Task 2.4 (optional demo):** Extract a `query_embedding` (section 5) for sanity checks / timing demos
- **Task 2.5:** Compare each crop embedding to the **gallery** (`cosine_distance`; **vote max (MIN_VOTES floor, then argmax count)** in 6.5 / 8)
- **Task 2.6:** Assign ID using **voting + `MIN_VOTES`** (per face), not plain nearest gallery row
- **Task 2.7:** Draw bounding boxes + labels — section 6.5 (and metrics + CSV in 8)

This notebook aligns with `FUTURE_IDEAS.md` (votes under a distance threshold; pick identity with most evidence among those meeting a minimum count).
